In [1]:
import requests
import pandas as pd
from pathlib import Path
import time 

In [2]:
# Atlantic provinces
provinces = ["NS", "NB", "PE", "NL"]

# Base URL for monthly climate observations
base_url = "https://dd.weather.gc.ca/today/climate/observations/monthly/csv"

# Local directory to store raw files
raw_dir = Path("../Capstone/raw data/precipitation_monthly")
raw_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def download_csv(url, output_path, wait=2):
    if not output_path.exists():
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                output_path.write_bytes(response.content)
                time.sleep(wait) 
            else:
                print(f"Failed: {url}")
                time.sleep(wait)
        except requests.exceptions.RequestException as e:
            print(f"Connection issue: {url}")
            time.sleep(wait * 2)  

In [4]:
import re

def list_csv_files(province):
    index_url = f"{base_url}/{province}/"
    response = requests.get(index_url)
    files = re.findall(r'href="(climate_monthly_.*?\.csv)"', response.text)
    return files

for prov in provinces:
    prov_dir = raw_dir / prov
    prov_dir.mkdir(exist_ok=True)
    
    files = list_csv_files(prov)
    print(f"{prov}: {len(files)} files found")
    
    for file_name in files:
        file_url = f"{base_url}/{prov}/{file_name}"
        output_file = prov_dir / file_name
        download_csv(file_url, output_file)

NS: 7246 files found
NB: 5556 files found
PE: 1294 files found
NL: 6966 files found


In [5]:
all_dfs = []

for prov in provinces:
    prov_dir = raw_dir / prov
    for csv_file in prov_dir.glob("*.csv"):
        df = pd.read_csv(csv_file)
        df["province"] = prov
        all_dfs.append(df)

precip_raw = pd.concat(all_dfs, ignore_index=True)

C:\Users\biauser\AppData\Local\Temp\ipykernel_3156\2751649746.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  precip_raw = pd.concat(all_dfs, ignore_index=True)


In [6]:
precip_raw.info()
precip_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224734 entries, 0 to 224733
Data columns (total 28 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Station Name  224734 non-null  object 
 1   Longitude     223786 non-null  float64
 2   Latitude      223786 non-null  float64
 3   Climate ID    224734 non-null  object 
 4   Province      224734 non-null  object 
 5   Year          224734 non-null  object 
 6   Month         224734 non-null  object 
 7   Tm            197882 non-null  float64
 8   DwTm          197882 non-null  float64
 9   D             78710 non-null   float64
 10  Tx            198180 non-null  float64
 11  DwTx          198180 non-null  float64
 12  Tn            198218 non-null  float64
 13  DwTn          198218 non-null  float64
 14  S             192066 non-null  float64
 15  DwS           192066 non-null  float64
 16  S%N           54191 non-null   object 
 17  P             216179 non-null  float64
 18  DwP 

,Station Name,Longitude,Latitude,Climate ID,Province,Year,Month,Tm,DwTm,D,...,DwP,P%N,S_G,Pd,BS,DwBS,BS%,HDD,CDD,province
0,ABERCROMBIE POINT,-62.717,45.65,8200015,NS,1973,9,13.2,0.0,NaN,...,0.0,NaN,NaN,5.0,NaN,NaN,NaN,149.1,5.9,NS
1,ABERCROMBIE POINT,-62.717,45.65,8200015,NS,1973,10,8.0,0.0,NaN,...,0.0,NaN,NaN,14.0,NaN,NaN,NaN,310.0,0.0,NS
2,ABERCROMBIE POINT,-62.717,45.65,8200015,NS,1973,11,2.0,1.0,NaN,...,0.0,NaN,NaN,12.0,NaN,NaN,NaN,463.0,0.0,NS
3,ABERCROMBIE POINT,-62.717,45.65,8200015,NS,1973,12,1.4,1.0,NaN,...,0.0,NaN,NaN,16.0,NaN,NaN,NaN,497.8,0.0,NS
4,ABERCROMBIE POINT,-62.717,45.65,8200015,NS,1974,1,-7.9,1.0,NaN,...,0.0,NaN,NaN,11.0,NaN,NaN,NaN,775.6,0.0,NS


In [7]:
# Keep only years relevant to the project scope
precip_raw = precip_raw[
    (precip_raw["Year"] >= 1995) & (precip_raw["Year"] <= 2025)
]

In [8]:
output_path = Path("../Capstone/raw data/precipitation_monthly_raw_atlantic.csv")
output_path.parent.mkdir(exist_ok=True)

precip_raw.to_csv(output_path, index=False)